# RenewCast: Renewable Generation Forecasting

**Research prototype for solar and wind power forecasting using the NREL ARPA-E PERFORM dataset.**

This notebook loads real HDF5 (`.h5`) actual-generation files, validates and prepares time series, then compares:
1. Persistence baseline
2. Gradient-boosted trees
3. LSTM sequence model (if TensorFlow is available)

It evaluates each technology separately using a chronological train/validation/test split and reports MAE, RMSE, and normalized MAE. It also exports metrics, predictions, plots, and trained models.

**Important scope note:** this version forecasts generation from historical generation and calendar features. It does **not** claim to be weather-aware unless genuine weather/forecast variables are added as input features. PERFORM also contains forecast files, but those have a different schema and should be handled in a separate experiment to avoid accidentally using future information.

**Dataset documentation:** https://github.com/PERFORM-Forecasts/documentation  
**Dataset DOI:** https://dx.doi.org/10.25984/1891136

Run cells from top to bottom in Google Colab. Upload one solar actuals `.h5` file and/or one wind actuals `.h5` file when prompted.

In [ ]:
# 1. Install dependencies
%pip -q install h5py scikit-learn pandas numpy matplotlib joblib

In [ ]:
# 2. Imports and reproducibility
import os, json, math, warnings, shutil
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings("ignore")

OUTPUT_DIR = Path("renewcast_outputs")
MODEL_DIR = OUTPUT_DIR / "models"
PLOT_DIR = OUTPUT_DIR / "plots"
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

print("Output directory:", OUTPUT_DIR.resolve())

## 3. Load the NREL PERFORM actuals files

PERFORM actuals files are HDF5 and commonly contain:
- `actuals`: array shaped `(time_steps, sites)`
- `time_index`: timestamps
- `meta`: site metadata

Upload actual-generation files, not forecast files, for this first experiment. If using Colab, the upload dialog appears in the next cell. You can upload one or both technologies.

In [ ]:
# In Google Colab, upload solar/wind actuals .h5 files.
# Outside Colab, place files in the working directory and skip this cell.
uploaded_files = {}
try:
    from google.colab import files
    uploaded_files = files.upload()
    print("Uploaded:", list(uploaded_files.keys()))
except Exception:
    print("Not running in Google Colab. Place .h5 files in the current directory.")

In [ ]:
# 4. Inspect HDF5 structure before modelling
def inspect_h5(path):
    print(f"\\nFILE: {path}")
    with h5py.File(path, "r") as h5:
        def visitor(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  {name}: shape={obj.shape}, dtype={obj.dtype}")
        h5.visititems(visitor)

h5_files = sorted([str(p) for p in Path(".").glob("*.h5")] + [str(p) for p in Path(".").glob("*.hdf5")])
if not h5_files:
    print("No .h5/.hdf5 files found. Upload the NREL PERFORM actuals files and rerun this cell.")
else:
    for f in h5_files:
        inspect_h5(f)

In [ ]:
# 5. HDF5 reader with timestamp and site selection
def _decode_time_index(raw):
    arr = np.asarray(raw).reshape(-1)
    # Handle numeric Unix timestamps in seconds or milliseconds.
    if np.issubdtype(arr.dtype, np.number):
        vals = arr.astype("float64")
        med = np.nanmedian(vals)
        unit = "ms" if med > 1e11 else "s"
        dt = pd.to_datetime(vals, unit=unit, utc=True, errors="coerce")
        # If values are small numeric counters rather than Unix timestamps, fail clearly.
        if dt.isna().mean() > 0.5 or dt.year.min() < 1990:
            raise ValueError("time_index is numeric but does not look like Unix timestamps. Inspect file metadata and adapt timestamp parsing.")
        return pd.DatetimeIndex(dt)
    # Handle byte strings / text timestamps.
    decoded = []
    for x in arr:
        if isinstance(x, (bytes, np.bytes_)):
            decoded.append(x.decode("utf-8", errors="replace"))
        else:
            decoded.append(str(x))
    dt = pd.to_datetime(decoded, utc=True, errors="coerce")
    if pd.isna(dt).mean() > 0.5:
        raise ValueError("Could not parse time_index as timestamps.")
    return pd.DatetimeIndex(dt)

def load_perform_actuals(path, site_index=0):
    with h5py.File(path, "r") as h5:
        required = {"actuals", "time_index"}
        missing = required.difference(h5.keys())
        if missing:
            raise ValueError(f"{path} is missing required datasets: {missing}. Is this an actuals file?")
        actuals = np.asarray(h5["actuals"])
        times = _decode_time_index(h5["time_index"][:])
        if actuals.ndim == 1:
            values = actuals.astype("float64")
        elif actuals.ndim == 2:
            if site_index >= actuals.shape[1]:
                raise IndexError(f"site_index={site_index}, but file has {actuals.shape[1]} columns")
            values = actuals[:, site_index].astype("float64")
        else:
            raise ValueError(f"Unsupported actuals array shape: {actuals.shape}")
        if len(times) != len(values):
            raise ValueError(f"Timestamp count {len(times)} does not match value count {len(values)}")
        frame = pd.DataFrame({"timestamp": times, "power": values})
    frame = frame.replace([np.inf, -np.inf], np.nan).dropna()
    frame = frame.sort_values("timestamp").drop_duplicates("timestamp", keep="last")
    frame = frame.set_index("timestamp")
    # Keep values as supplied; don't assume whether they are MW, kW, or capacity factor.
    return frame

print("Reader ready.")

## 4. Choose files and configure the experiment

Set the filenames below to match the uploaded file names. Use actuals files for solar and wind separately. If the file is very large, start with one system-level or zone-level file, or a manageable site-level file.

**Forecast horizon:** one time step ahead. With PERFORM actuals at 5-minute resolution, this means 5 minutes ahead. Change `HORIZON_STEPS` to 12 for a one-hour-ahead target on 5-minute data.

The sequence window is the past 12 time steps by default. This is only 1 hour for 5-minute data, so increase it if justified by the sampling frequency and available compute.

In [ ]:
# EDIT THESE FILE NAMES after inspecting the files in section 4.
# Use None if you are not running that technology yet.
SOLAR_FILE = None  # e.g. "solar_actuals.h5"
WIND_FILE  = None  # e.g. "wind_actuals.h5"

# Site/column to model from each file. For a system-level file with one column, use 0.
SOLAR_SITE_INDEX = 0
WIND_SITE_INDEX = 0

HORIZON_STEPS = 1
LOOKBACK_STEPS = 12
TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
# Remaining 15% is held out as the final test period.

print("Available files:", h5_files)
print("Set SOLAR_FILE and/or WIND_FILE to an exact filename above.")

In [ ]:
# 7. Feature engineering and chronological split
def make_supervised(frame, horizon_steps=1, lookback_steps=12):
    df = frame.copy()
    y_future = df["power"].shift(-horizon_steps)
    X = pd.DataFrame(index=df.index)

    # Calendar features known at forecast issue time
    idx = df.index
    X["hour_sin"] = np.sin(2 * np.pi * (idx.hour + idx.minute / 60) / 24)
    X["hour_cos"] = np.cos(2 * np.pi * (idx.hour + idx.minute / 60) / 24)
    X["dayofyear_sin"] = np.sin(2 * np.pi * idx.dayofyear / 365.25)
    X["dayofyear_cos"] = np.cos(2 * np.pi * idx.dayofyear / 365.25)
    X["dayofweek_sin"] = np.sin(2 * np.pi * idx.dayofweek / 7)
    X["dayofweek_cos"] = np.cos(2 * np.pi * idx.dayofweek / 7)

    # Only past observations are used as inputs.
    for lag in range(lookback_steps):
        X[f"lag_{lag}"] = df["power"].shift(lag)
    # Past-only rolling summaries
    X["rolling_mean_3"] = df["power"].shift(1).rolling(3, min_periods=3).mean()
    X["rolling_mean_12"] = df["power"].shift(1).rolling(12, min_periods=12).mean()
    X["rolling_std_12"] = df["power"].shift(1).rolling(12, min_periods=12).std()

    out = X.copy()
    out["target"] = y_future
    out = out.replace([np.inf, -np.inf], np.nan).dropna()
    return out.drop(columns="target"), out["target"]

def chronological_split(X, y, train_fraction=0.70, val_fraction=0.15):
    n = len(X)
    train_end = int(n * train_fraction)
    val_end = int(n * (train_fraction + val_fraction))
    if train_end < 100 or val_end <= train_end or val_end >= n:
        raise ValueError(f"Not enough usable rows ({n}) for this split.")
    return (
        X.iloc[:train_end], y.iloc[:train_end],
        X.iloc[train_end:val_end], y.iloc[train_end:val_end],
        X.iloc[val_end:], y.iloc[val_end:]
    )

def metric_row(y_true, y_pred, model_name, scale):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    nmae = mae / scale * 100 if scale and np.isfinite(scale) and scale > 0 else np.nan
    return {"model": model_name, "MAE": mae, "RMSE": rmse, "nMAE_percent": nmae}

def run_tree_experiment(frame, technology):
    X, y = make_supervised(frame, HORIZON_STEPS, LOOKBACK_STEPS)
    Xtr, ytr, Xv, yv, Xte, yte = chronological_split(X, y, TRAIN_FRACTION, VALIDATION_FRACTION)

    # Select model configuration on validation data only; final evaluation is test-only.
    candidates = [
        HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_leaf_nodes=15,
                                      l2_regularization=1.0, random_state=SEED),
        HistGradientBoostingRegressor(max_iter=250, learning_rate=0.03, max_leaf_nodes=20,
                                      l2_regularization=2.0, random_state=SEED),
    ]
    best_model, best_val_mae = None, float("inf")
    for candidate in candidates:
        candidate.fit(Xtr, ytr)
        val_pred = candidate.predict(Xv)
        val_mae = mean_absolute_error(yv, val_pred)
        if val_mae < best_val_mae:
            best_model, best_val_mae = candidate, val_mae

    # Retrain on train + validation after model configuration selection.
    X_trainval = pd.concat([Xtr, Xv])
    y_trainval = pd.concat([ytr, yv])
    best_model.fit(X_trainval, y_trainval)
    tree_pred = best_model.predict(Xte)

    # Persistence forecast: last observed power value at forecast issue time.
    persistence_pred = Xte["lag_0"].to_numpy()
    scale = float(np.nanmax(y_trainval) - np.nanmin(y_trainval))
    rows = [
        metric_row(yte, persistence_pred, "Persistence", scale),
        metric_row(yte, tree_pred, "GradientBoosting", scale)
    ]
    metrics = pd.DataFrame(rows)
    predictions = pd.DataFrame({
        "timestamp": yte.index,
        "actual": yte.to_numpy(),
        "persistence": persistence_pred,
        "gradient_boosting": tree_pred
    }).set_index("timestamp")

    stem = technology.lower().replace(" ", "_")
    joblib.dump(best_model, MODEL_DIR / f"{stem}_gradient_boosting.joblib")
    metrics.to_csv(OUTPUT_DIR / f"{stem}_metrics.csv", index=False)
    predictions.to_csv(OUTPUT_DIR / f"{stem}_predictions.csv")

    plt.figure(figsize=(13, 4))
    view = predictions.iloc[:min(1000, len(predictions))]
    plt.plot(view.index, view["actual"], label="Actual", linewidth=1.4)
    plt.plot(view.index, view["persistence"], label="Persistence", alpha=0.8)
    plt.plot(view.index, view["gradient_boosting"], label="Gradient boosting", alpha=0.8)
    plt.title(f"{technology}: held-out test forecasts")
    plt.xlabel("Time (UTC)")
    plt.ylabel("Power (units as provided by dataset)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{stem}_test_forecasts.png", dpi=160)
    plt.show()

    return metrics, predictions, (Xtr, ytr, Xv, yv, Xte, yte), best_model

print("Feature and evaluation functions ready.")

In [ ]:
# 8. Run baseline + tree model for solar and/or wind
experiments = {}
for technology, filename, site_idx in [
    ("Solar", SOLAR_FILE, SOLAR_SITE_INDEX),
    ("Wind", WIND_FILE, WIND_SITE_INDEX)
]:
    if filename is None:
        print(f"Skipping {technology}: set its filename in section 6.")
        continue
    if not Path(filename).exists():
        print(f"Skipping {technology}: file not found: {filename}")
        continue
    print(f"\\nLoading {technology}: {filename}")
    frame = load_perform_actuals(filename, site_index=site_idx)
    print("Rows:", len(frame), "| Range:", frame.index.min(), "to", frame.index.max())
    print(frame["power"].describe())
    metrics, predictions, split_data, model = run_tree_experiment(frame, technology)
    experiments[technology] = {
        "frame": frame, "metrics": metrics, "predictions": predictions,
        "split_data": split_data, "tree_model": model
    }
    display(metrics)

if not experiments:
    print("No experiments ran yet. Set SOLAR_FILE and/or WIND_FILE to exact filenames from section 4, then rerun.")

## 5. Optional LSTM comparison

This section trains an LSTM on the same chronological series. It uses only past generation values and calendar features, so it remains a historical-generation forecasting model rather than a weather-aware model.

The LSTM is evaluated on the same test interval. The model is intentionally small to fit free Colab compute.

In [ ]:
# 9. LSTM training and test evaluation (requires TensorFlow)
def run_lstm_experiment(frame, technology):
    try:
        import tensorflow as tf
        from tensorflow.keras import layers, models, callbacks
    except Exception as e:
        print("TensorFlow unavailable; skipping LSTM:", e)
        return None

    tf.keras.utils.set_random_seed(SEED)
    # Use a single series and windowed past values to avoid feature/target leakage.
    values = frame["power"].astype("float32").to_numpy()
    times = frame.index
    n = len(values)
    train_end = int(n * TRAIN_FRACTION)
    val_end = int(n * (TRAIN_FRACTION + VALIDATION_FRACTION))

    # Scale using training period only.
    mean = float(np.nanmean(values[:train_end]))
    std = float(np.nanstd(values[:train_end]))
    if not np.isfinite(std) or std == 0:
        raise ValueError("Training data has zero/invalid variance.")
    scaled = (values - mean) / std

    def make_windows(start_target, end_target):
        Xw, yw, tw = [], [], []
        for target_i in range(max(start_target, LOOKBACK_STEPS), min(end_target, n - HORIZON_STEPS)):
            Xw.append(scaled[target_i-LOOKBACK_STEPS:target_i])
            yw.append(scaled[target_i + HORIZON_STEPS - 1])
            tw.append(times[target_i + HORIZON_STEPS - 1])
        return np.asarray(Xw)[..., None], np.asarray(yw), pd.DatetimeIndex(tw)

    X_train, y_train, _ = make_windows(LOOKBACK_STEPS, train_end - HORIZON_STEPS)
    X_val, y_val, _ = make_windows(train_end, val_end - HORIZON_STEPS)
    X_test, y_test, t_test = make_windows(val_end, n - HORIZON_STEPS)
    if len(X_train) == 0 or len(X_val) == 0 or len(X_test) == 0:
        raise ValueError("Not enough data for LSTM splits.")

    model = models.Sequential([
        layers.Input(shape=(LOOKBACK_STEPS, 1)),
        layers.LSTM(32),
        layers.Dense(16, activation="relu"),
        layers.Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=40, batch_size=128, callbacks=[early_stop], verbose=1)

    # Test targets converted back to original units.
    pred_scaled = model.predict(X_test, verbose=0).reshape(-1)
    pred = pred_scaled * std + mean
    actual = y_test * std + mean
    persistence = values[np.asarray([times.get_indexer([t])[0] for t in t_test]) - HORIZON_STEPS]
    scale = float(np.nanmax(values[:val_end]) - np.nanmin(values[:val_end]))
    metrics = pd.DataFrame([
        metric_row(actual, persistence, "Persistence", scale),
        metric_row(actual, pred, "LSTM", scale)
    ])
    predictions = pd.DataFrame({
        "timestamp": t_test, "actual": actual,
        "persistence": persistence, "lstm": pred
    }).set_index("timestamp")

    stem = technology.lower().replace(" ", "_")
    model.save(MODEL_DIR / f"{stem}_lstm.keras")
    metrics.to_csv(OUTPUT_DIR / f"{stem}_lstm_metrics.csv", index=False)
    predictions.to_csv(OUTPUT_DIR / f"{stem}_lstm_predictions.csv")

    plt.figure(figsize=(13, 4))
    view = predictions.iloc[:min(1000, len(predictions))]
    plt.plot(view.index, view["actual"], label="Actual", linewidth=1.4)
    plt.plot(view.index, view["persistence"], label="Persistence", alpha=0.8)
    plt.plot(view.index, view["lstm"], label="LSTM", alpha=0.8)
    plt.title(f"{technology}: LSTM held-out test forecasts")
    plt.xlabel("Time (UTC)")
    plt.ylabel("Power (units as provided by dataset)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(PLOT_DIR / f"{stem}_lstm_test_forecasts.png", dpi=160)
    plt.show()
    return metrics, predictions, model

lstm_experiments = {}
for technology, result in experiments.items():
    try:
        out = run_lstm_experiment(result["frame"], technology)
        if out is not None:
            lstm_experiments[technology] = out
            display(out[0])
    except Exception as e:
        print(f"LSTM failed for {technology}: {type(e).__name__}: {e}")

## 6. Error analysis and exports

The final test period is the only period used for the headline result. Review the model against persistence before drawing conclusions. A negative error reduction means the model performed worse than persistence.

Normalized MAE is normalized by the training/validation range in this prototype; state that definition explicitly in the report. It is not the same as capacity-normalized MAE unless rated capacity is available from the metadata.

In [ ]:
# 10. Combined summary and error reduction vs persistence
all_rows = []
for technology, result in experiments.items():
    m = result["metrics"].copy()
    m.insert(0, "technology", technology)
    all_rows.append(m)
for technology, (m, predictions, model) in lstm_experiments.items():
    m = m.copy()
    m.insert(0, "technology", technology)
    all_rows.append(m)

if all_rows:
    summary = pd.concat(all_rows, ignore_index=True)
    # Add percentage error reduction relative to persistence for each technology.
    summary["error_reduction_vs_persistence_percent"] = np.nan
    for tech in summary["technology"].unique():
        mask = summary["technology"] == tech
        baseline = summary.loc[mask & (summary["model"] == "Persistence"), "MAE"]
        if len(baseline) and float(baseline.iloc[0]) != 0:
            base_mae = float(baseline.iloc[0])
            summary.loc[mask, "error_reduction_vs_persistence_percent"] = (
                (base_mae - summary.loc[mask, "MAE"]) / base_mae * 100
            )
    display(summary.sort_values(["technology", "MAE"]))
    summary.to_csv(OUTPUT_DIR / "combined_metrics.csv", index=False)
else:
    print("No results available yet.")

In [ ]:
# 11. Generate a transparent experiment report
report = {
    "project": "RenewCast",
    "dataset": "NREL ARPA-E PERFORM actual-generation HDF5 files",
    "dataset_documentation": "https://github.com/PERFORM-Forecasts/documentation",
    "forecast_horizon_steps": HORIZON_STEPS,
    "lookback_steps": LOOKBACK_STEPS,
    "split": {
        "train_fraction": TRAIN_FRACTION,
        "validation_fraction": VALIDATION_FRACTION,
        "test_fraction": round(1 - TRAIN_FRACTION - VALIDATION_FRACTION, 4),
        "method": "chronological"
    },
    "feature_scope": "past generation lags, rolling summaries, calendar variables; no weather variables in this baseline notebook",
    "technologies_run": list(experiments.keys()),
    "results": summary.to_dict(orient="records") if all_rows else [],
    "limitations": [
        "This notebook forecasts generation from historical generation and calendar features; it is not yet a weather-aware model.",
        "PERFORM generation profiles are simulated/modelled in part from meteorological inputs, not necessarily direct meter readings.",
        "One selected site/column may not represent an entire region or grid.",
        "No operational grid deployment is performed.",
        "Results apply to the selected dataset, site/column, time period, horizon, and split."
    ]
}
with open(OUTPUT_DIR / "experiment_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)
print(json.dumps(report, indent=2, default=str)[:12000])

In [ ]:
# 12. Package all outputs into a ZIP for download
zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Created:", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download the ZIP from the notebook's file browser.")

## 7. Next research step: add genuine weather forecasts

Do not label this baseline notebook weather-aware. To test the original hypothesis, the next version must align weather forecasts **available at the forecast issue time** with future generation, for example:
- solar: forecast irradiance, cloud cover, temperature;
- wind: forecast wind speed/direction, temperature and pressure.

For PERFORM, inspect the forecast file documentation and `issue_time` / `forecast_time` fields before joining. Use the forecast issued before the target timestamp, not weather observations that would only be known afterward. Compare:
1. persistence;
2. PERFORM's supplied generation forecast, where compatible;
3. historical-generation ML;
4. weather-aware ML;
5. LSTM.

That experiment can support a defensible claim about the value of weather inputs. Cite the dataset and report the actual held-out results, including cases where the new model does not improve performance.